# **MASTER BUDDY - AI**

## StudyBuddy-AI (DPO Preference-Aligned Study Assistant)

## Overview

StudyBuddy-AI is an AI-powered educational assistant designed to help students understand complex topics in a **clear, structured, and beginner-friendly way**. Unlike standard chatbots, this project focuses on **preference-aligned learning using Direct Preference Optimization (DPO)** to ensure the model consistently produces high-quality explanations.

The system is trained to prefer answers that are:
- Simple and easy to understand
- Well-structured with step-by-step reasoning
- Supported with examples where possible
- Free from unnecessary technical jargon

---

## Objective

The main goal of this project is to build a **study assistant that learns from human preferences**, specifically teaching the model:

> “Good explanations should feel like a patient tutor, not a technical manual.”

---

## Core Concept

This project uses **Direct Preference Optimization (DPO)** to fine-tune a language model based on comparison pairs:

For each question:
- **Prompt** → Student question
- **Chosen response** → Clear, structured, student-friendly explanation
- **Rejected response** → Confusing, overly technical, or poorly structured answer

This allows the model to learn *what makes a good educational explanation*.

---

## Example Training Pair

**Prompt:**
> Explain TCP vs UDP

**Chosen Answer:**
- Simple comparison
- Clear structure
- Real-world examples

**Rejected Answer:**
- Dense technical jargon
- No formatting
- Hard to understand

---

## Tech Stack

- 🧠 Hugging Face Transformers  
- ⚙️ TRL (DPOTrainer)  
- 🤖 LLaMA / Mistral small models  
- 📊 Custom preference dataset  
- 🐍 Python (PyTorch backend)

---

## Key Features

- Preference-aligned fine-tuning using DPO
- Educational chatbot behavior optimization
- Dataset creation for instruction + preference pairs
- Lightweight and scalable training pipeline
- Focus on real-world student learning experience

---

## Expected Outcome

After training, StudyBuddy-AI will:
- Generate clearer explanations than base models
- Prefer structured teaching-style responses
- Act like a **personal AI tutor**

---

## Future Improvements

- Add multi-language support
- Integrate RAG for textbook knowledge
- Expand dataset with real student feedback
- Deploy as web app (React + FastAPI)

---

## Why This Project Matters

This project demonstrates how **AI alignment techniques like DPO** can directly improve real-world usability, especially in education where clarity is more important than complexity.

#### Import Libraries

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer
from datasets import load_dataset, Dataset
import json
import dotenv
from huggingface_hub import login
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import train_test_split

##### Helper Function

In [2]:
HF_Token = dotenv.get_key("../.env", "HF_TOKEN")

In [3]:
login(HF_Token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
def get_dataset(filepath):
    data = []
    
    with open(filepath, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

In [27]:
dataset = get_dataset("../dataset/dpo_study_assistant_1000.jsonl")
dataset[0:10]

[{'prompt': 'Give a simple explanation of database normalization',
  'chosen': 'Normalization is organizing database tables to reduce redundancy and improve integrity. This is important for exams and practical understanding.',
  'rejected': 'Normalization is used in databases.'},
 {'prompt': 'Describe encryption vs hashing',
  'chosen': 'Encryption is reversible and used to protect data in transit or at rest, while hashing is one-way and used to verify data integrity or store passwords securely. This is important for exams and practical understanding.',
  'rejected': 'Encryption and hashing are security techniques.'},
 {'prompt': 'Give a simple explanation of encryption vs hashing',
  'chosen': 'Encryption is reversible and used to protect data in transit or at rest, while hashing is one-way and used to verify data integrity or store passwords securely. This is important for exams and practical understanding.',
  'rejected': 'Encryption and hashing are security techniques.'},
 {'prompt

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device.type

'cuda'

In [7]:
#model_name = "mistralai/Mistral-7B-Instruct-v0.2"
model_name = "meta-llama/Llama-3.2-1B-Instruct"

In [8]:
model = AutoModelForCausalLM.from_pretrained(model_name)
model.to(device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
ref_model = AutoModelForCausalLM.from_pretrained(model_name)
ref_model.to(device)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [12]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [28]:
dataset[0]

{'prompt': 'Give a simple explanation of database normalization',
 'chosen': 'Normalization is organizing database tables to reduce redundancy and improve integrity. This is important for exams and practical understanding.',
 'rejected': 'Normalization is used in databases.'}

In [17]:
def format_prompt (dataset):
    
    for data in dataset:
        message = [{"role":"user", "content": data['prompt']}]
        prompt = tokenizer.apply_chat_template(
            message,
            tokenize = False,
            add_generation_prompt = True
        )
        
        data['prompt'] = prompt
    return dataset

In [29]:
dataset_format = format_prompt(dataset)

In [30]:
dataset_format[0:4]

[{'prompt': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 03 Jun 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nGive a simple explanation of database normalization<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n',
  'chosen': 'Normalization is organizing database tables to reduce redundancy and improve integrity. This is important for exams and practical understanding.',
  'rejected': 'Normalization is used in databases.'},
 {'prompt': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 03 Jun 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nDescribe encryption vs hashing<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n',
  'chosen': 'Encryption is reversible and used to protect data in transit or at rest, while hashing is one-way and used to verify data integrity or store passwords securely. This is importa

In [32]:
test_size = int(len(dataset_format)*0.2)
train_size = len(dataset_format)-test_size

train_set, test_set = train_test_split(dataset_format, test_size=test_size, train_size=train_size, random_state=42, shuffle=True)

In [33]:
len(train_set), len(test_set)

(800, 200)

In [38]:
train_set = Dataset.from_list(train_set)
test_set = Dataset.from_list(test_set)

### QLoRA, LoRA and DPO Configuration

In [23]:
qlora_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

qlora_model = AutoModelForCausalLM.from_pretrained(model_name, 
                                                quantization_config = qlora_config)

W0603 12:33:04.715000 25032 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [24]:
qlora_model.config.use_cache = False
qlora_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm

In [26]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=4,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']
)

peft_model = get_peft_model(model=qlora_model, peft_config=lora_config)
peft_model.to(device)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.L

In [34]:
dpo_config = DPOConfig(
    beta=0.1,
    output_dir="/model_output",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    learning_rate=1e-4,
    logging_steps=40,
    eval_steps=100,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
)

In [39]:
dpo_trainer = DPOTrainer(
    model=peft_model,
    ref_model=ref_model,
    args=dpo_config,
    train_dataset=train_set,
    eval_dataset=test_set,
    processing_class=tokenizer
)

Adding EOS to train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

In [40]:
dpo_trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [7]:
model_name= "SeranVishwa/master-buddy-v1.0"

In [2]:
model = AutoModelForCausalLM.from_pretrained("SeranVishwa/master-buddy-v1.0")
tokenizer = AutoTokenizer.from_pretrained("SeranVishwa/master-buddy-v1.0")

config.json:   0%|          | 0.00/934 [00:00<?, ?B/s]

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vish\.cache\huggingface\hub\models--SeranVishwa--master-buddy-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [37]:
from langchain_core.prompts import PromptTemplate,ChatMessagePromptTemplate
from langchain_huggingface import ChatHuggingFace,HuggingFacePipeline
from langchain_core.messages import HumanMessage,AIMessage
from langchain_classic.chains import LLMChain
from langchain_core.output_parsers import StrOutputParser
import re

In [60]:
llm = HuggingFacePipeline.from_model_id(
    model_id="SeranVishwa/master-buddy-v1.0",
    task="text-generation",
    pipeline_kwargs={"max_new_tokens":2048,"max_length": None}
   
)

llm_model = ChatHuggingFace(llm=llm, temperature=1)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [61]:
template = """You are a helpful study assistant. give accurate answers descriptively for user questions.
                user: {question}
                response:
"""

prompt_template = PromptTemplate(template=template, 
                                input_variables=['question'])



In [62]:

def  clean_response (response):
    match = re.search(r'<\|start_header_id\|>assistant<\|end_header_id\|>\s*(.*)',response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return response.strip()


In [63]:
chain = prompt_template | llm_model | StrOutputParser()

In [65]:
response =  chain.invoke({"question": "explain each data structure component in very descriptively"})
print(clean_response(response))

I'm here to help you understand each data structure component in detail.

**1. Arrays**
An array is a collection of elements of the same data type stored in contiguous memory locations. It's denoted by square brackets `[]` and is denoted by the `length` property. Arrays are useful for storing a fixed-size collection of elements, such as exam scores or student grades.

Example:
```javascript
let scores = [90, 80, 70, 60];  // Create an array of 4 scores
console.log(scores[0]);  // Output: 90
```
**2. Objects**
An object is a collection of key-value pairs stored in memory as key-value pairs. It's denoted by curly brackets `{}`. Objects are useful for storing data that has a clear key-value relationship, such as student information or employee details.

Example:
```javascript
let student = { name: 'John', age: 20, address: '123 Main St' };
console.log(student.name);  // Output: John
```
**3. Sets**
A set is an unordered collection of unique elements. It's denoted by the `set` keyword and 

In [ ]:
response =  chain.invoke({"question": "explain each data structure component in very descriptively"})
print(clean_response(response))

I'll provide a detailed explanation of each data structure component, breaking down the concepts with descriptive examples.

### Arrays
#### What is an Array?
An array is a collection of elements of the same data type stored in contiguous memory locations. It's a fundamental data structure in programming that allows efficient insertion, deletion, and traversal of elements.

#### Key Components:
- **Index**: The position of an element in the array (0-based indexing).
- **Size**: The number of elements currently stored in the array.
- **Capacity**: The maximum number of elements the array can hold; it's initially 0.
- **Capacity Factor**: The ratio of current size to maximum capacity.
- **Size of New Element**: The space required for a new element.

#### Example:
```python
my_array = [1, 2, 3, 4, 5]
```
- Index: 0 (the first element)
- Size: 5 (4 elements)
- Capacity: 5 (initially 0)
- Capacity Factor: 1.0 (100% capacity utilization)
- Size of New Element: 5 (5 elements required for a ne